## Companion Notebook: How Much of a Data Science Workflow Can Run on a GPU Today? Accelerating Data Preparation
## Accelerating pandas code with cudf.pandas

This notebook shows how to run existing pandas-style code on the GPU with cudf.pandas, using the NYC Yellow Taxi dataset.

#### Setup

In Google Colab, use a GPU runtime before running this notebook. The cudf.pandas extension should be loaded before importing pandas.

In [1]:
!nvidia-smi

Sun Aug 23 19:12:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.62                 Driver Version: 592.01         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5080 ...    On  |   00000000:02:00.0 Off |                  N/A |
| N/A   41C    P8              5W /   95W |    4197MiB /  16303MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%load_ext cudf.pandas

import pandas as pd
import requests
from io import BytesIO

## Load the data

We use the first three months of 2023 NYC Yellow Taxi data.

In [3]:
def load_data_pandas(months=3):
    base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
    frames = []

    for month in range(1, months + 1):
        url = base_url.format(month)
        print(f"Downloading month {month:02d}...")
        response = requests.get(url)
        response.raise_for_status()
        df = pd.read_parquet(BytesIO(response.content))
        frames.append(df)

    return pd.concat(frames, ignore_index=True)

df = load_data_pandas(months=3)

print(f"Total rows: {len(df):,}")
print(f"Shape: {df.shape}")

Total rows: 9,384,487
Shape: (9384487, 20)


## Grouped summary

This is ordinary pandas code. With cudf.pandas loaded, supported operations can run on the GPU.

In [4]:
summary = (
    df.groupby("PULocationID")
      .agg(
          {
              "fare_amount": "sum",
              "trip_distance": "mean",
              "passenger_count": "count",
          }
      )
      .sort_values("fare_amount", ascending=False)
)

summary

,fare_amount,trip_distance,passenger_count
PULocationID,,,
132,2.781169e+07,15.597877,461176
138,1.195445e+07,9.690808,288185
161,6.670039e+06,2.659514,425737
230,5.510921e+06,3.170446,309416
237,5.441654e+06,1.809370,429782
...,...,...,...
245,1.176000e+02,4.961250,8
199,1.149000e+02,6.275000,4
176,8.550000e+01,3.800000,4


In [5]:
print(pd)

<module 'pandas' (ModuleAccelerator(fast=cudf, slow=pandas))>


## Profiling GPU and CPU Execution

In [6]:
%%cudf.pandas.profile
summary = (
    df.groupby("payment_type")
      .fare_amount.mean()
)

                                                                                                       
                                       Total time elapsed: 1.104 seconds                               
                                     2 GPU function calls in 0.949 seconds                             
                                     0 CPU function calls in 0.000 seconds                             
                                                                                                       
                                                     Stats                                             
                                                                                                       
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function          ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.groupby │ 1          │ 0.167       │ 0.167       │ 0          │ 0.000       │ 0.000       │
│ GroupBy.mean      │ 1          │ 0.782       │ 0.782       │ 0          │ 0.000       │ 0.000       │
└───────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [7]:
%%cudf.pandas.line_profile
summary = (
    df.groupby("payment_type")
      .fare_amount.mean()
)

                                                                             
                          Total time elapsed: 1.136 seconds                  
                                                                             
                                        Stats                                
                                                                             
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Line no. ┃ Line                               ┃ GPU TIME(s) ┃ CPU TIME(s) ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 1        │     summary = (                    │             │             │
│          │                                    │             │             │
│ 2        │         df.groupby("payment_type") │ 0.223123308 │             │
│          │                                    │             │             │
│ 3        │           .fare_amount.mean()      │ 0.751042812 │             │
│          │                                    │             │             │
└──────────┴────────────────────────────────────┴─────────────┴─────────────┘

In [9]:
!uv add plotly

Resolved 150 packages in 3.70s                                       
Prepared 2 packages in 494ms                                             
Installed 2 packages in 90ms                                
 + narwhals==2.25.0
 + plotly==6.9.0


In [10]:
%%cudf.pandas.profile
import plotly.express as px

top10 = summary.head(10).reset_index()

fig = px.bar(
    top10,
    x="payment_type",
    y="fare_amount",
    title="Average Fare by Payment Type",
    labels={
        "payment_type": "Payment Type",
        "fare_amount": "Average Fare",
    },
)

fig.show()

                                                                                                            
                                         Total time elapsed: 21.848 seconds                                 
                                       55 GPU function calls in 0.171 seconds                               
                                       8 CPU function calls in 0.004 seconds                                
                                                                                                            
                                                       Stats                                                
                                                                                                            
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function               ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ NDFrame.head           │ 1          │ 0.001       │ 0.001       │ 0          │ 0.000       │ 0.000       │
│ Series.reset_index     │ 1          │ 0.001       │ 0.001       │ 0          │ 0.000       │ 0.000       │
│ Index.__len__          │ 0          │ 0.000       │ 0.000       │ 2          │ 0.000       │ 0.000       │
│ DataFrame              │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ IndexOpsMixin.tolist   │ 0          │ 0.000       │ 0.000       │ 3          │ 0.000       │ 0.000       │
│ DataFrame.__getitem__  │ 6          │ 0.001       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ Series                 │ 2          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ Series.__len__         │ 37         │ 0.001       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__len__      │ 2          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ RangeIndex.__len__     │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ Series.set_axis        │ 0          │ 0.000       │ 0.000       │ 1          │ 0.001       │ 0.001       │
│ DataFrame.from_dict    │ 1          │ 0.004       │ 0.004       │ 0          │ 0.000       │ 0.000       │
│ IndexOpsMixin.to_numpy │ 0          │ 0.000       │ 0.000       │ 2          │ 0.002       │ 0.001       │
│ ndarray                │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ ndarray.copy           │ 2          │ 0.163       │ 0.081       │ 0          │ 0.000       │ 0.000       │
└────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- Index.__len__
- IndexOpsMixin.tolist
- Series.set_axis
- IndexOpsMixin.to_numpy

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=15395660;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.